<a href="https://colab.research.google.com/github/palakagarwal01/movie-recommendation/blob/main/Movie_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np

# Load datasets
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

# Merge both dataframes on the 'title' column
movies = movies.merge(credits, on='title')

# Select only the relevant features for content filtering
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
print("✅ Data Loaded and Merged Successfully! Current Shape:", movies.shape)
print(movies.head(1))


✅ Data Loaded and Merged Successfully! Current Shape: (4809, 7)
   movie_id   title                                           overview  \
0     19995  Avatar  In the 22nd century, a paraplegic Marine is di...   

                                              genres  \
0  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                            keywords  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...   

                                                cast  \
0  [{"cast_id": 242, "character": "Jake Sully", "...   

                                                crew  
0  [{"credit_id": "52fe48009251416c750aca23", "de...  


In [10]:
import ast

def convert_metadata(obj):
    """Extracts names from JSON-like string lists."""
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

# Clean genres and keywords
movies['genres'] = movies['genres'].apply(convert_metadata)
movies['keywords'] = movies['keywords'].apply(convert_metadata)

# Extract only the top 3 actors from the cast list
def convert_cast(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L
movies['cast'] = movies['cast'].apply(convert_cast)

# Extract only the Director from the crew list
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L
movies['crew'] = movies['crew'].apply(fetch_director)

# Clean string spaces (e.g., convert 'Johnny Depp' to 'JohnnyDepp' so the AI sees it as a single token)
def collapse_spaces(L):
    return [i.replace(" ", "") for i in L]

movies['genres'] = movies['genres'].apply(collapse_spaces)
movies['keywords'] = movies['keywords'].apply(collapse_spaces)
movies['cast'] = movies['cast'].apply(collapse_spaces)
movies['crew'] = movies['crew'].apply(collapse_spaces)

# Handle potential empty overviews
movies['overview'] = movies['overview'].fillna('').apply(lambda x: x.split())

print("✅ Meta-tags metadata processed successfully!")


✅ Meta-tags metadata processed successfully!


In [11]:
# Merge all text lists into a unified column
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# Create the finalized dataframe
new_df = movies[['movie_id', 'title', 'tags']]

# Convert the word lists back into clean paragraph strings
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x).lower())

print("✅ Master tags collection generated!")
print(new_df.head(1)['tags'].values[0][:150] + "...")


✅ Master tags collection generated!
in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecti...


/tmp/ipykernel_530/2409816145.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x).lower())


In [12]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Transform text paragraphs into numbers (Limit to top 5,000 most frequent words)
cv = CountVectorizer(max_features=5000, stop_words='english')
vector_matrix = cv.fit_transform(new_df['tags']).toarray()

# 2. Calculate the geometric similarity angle between all movie vectors
similarity = cosine_similarity(vector_matrix)

print("✅ Similarity Matrix Matrix Computed! Shape:", similarity.shape)


✅ Similarity Matrix Matrix Computed! Shape: (4809, 4809)


In [15]:
def recommend(movie_title):
    try:
        # 1. Safeguard copy array tracking matrix to fix SettingWithCopyWarning
        global new_df
        new_df = new_df.copy()

        # 2. Advanced Normalization: Strip characters & remove hyphens so "spider man" matches "Spider-Man"
        # We replace both hyphens and spaces to create a single-token matching view
        new_df['title_search'] = new_df['title'].astype(str).str.lower().str.replace('-', '', regex=False).str.replace(' ', '', regex=False)
        clean_input = movie_title.lower().replace('-', '').replace(' ', '')

        # 3. Dynamic Lookup
        search_results = new_df[new_df['title_search'].str.contains(clean_input, na=False)]

        if search_results.empty:
            print(f"❌ '{movie_title}' not found in the dataset matrix layers.")
            return

        # 4. Pull target index attributes safely using .iloc positioning structures
        matched_title = search_results.iloc[0]['title']
        movie_index = search_results.index[0]

        # 5. Extract Cosine Similarity weights matrix
        distances = similarity[movie_index]

        # Sort and filter the top 5 relative indices
        movies_list = sorted(list(enumerate(distances)), key=lambda x: x[1], reverse=True)[1:6]

        print(f"🍿 Found closest match in database: '{matched_title}'")
        print(f"🎬 Top 5 Recommendations for your profile:\n")
        print("=" * 60)
        for i in movies_list:
            print(f"🎥 {new_df.iloc[i[0]]['title']}")
        print("=" * 60)

    except Exception as e:
        print(f"❌ Runtime Pipeline Exception: {e}")

# 🚀 Run the ultimate search engine now!
recommend("spider man")


🍿 Found closest match in database: 'Spider-Man 3'
🎬 Top 5 Recommendations for your profile:

🎥 Spider-Man 2
🎥 Spider-Man
🎥 The Amazing Spider-Man 2
🎥 The Amazing Spider-Man
🎥 Arachnophobia


In [8]:
recommend("spider man")

❌ 'spider man' not found anywhere in the dataset. Check for typos!


In [14]:
def recommend(movie_title):
    try:
        # 1. Clean up any hidden spaces or weird formatting in the dataset strings
        new_df['title_clean'] = new_df['title'].astype(str).str.strip()

        # 2. Smart Match Attempt
        search_results = new_df[new_df['title_clean'].str.lower().str.contains(movie_title.lower())]

        # 3. Debug Trigger if search turns up empty
        if search_results.empty:
            print(f"❌ '{movie_title}' not found anywhere in the dataset.")
            print("\n🔍 Let's debug. Here are all titles in your dataset containing 'spider':")
            print("=" * 60)
            debug_matches = new_df[new_df['title'].astype(str).str.lower().str.contains('spider')]
            if not debug_matches.empty:
                print(debug_matches['title'].values)
            else:
                print("No movies containing the word 'spider' found at all. Did you run your previous data-loading cells?")
            print("=" * 60)
            return

        # 4. Pull indices if match is confirmed
        matched_title = search_results.iloc[0]['title']
        movie_index = search_results.index[0]

        # 5. Pipeline Cosine Similarity evaluation extraction
        distances = similarity[movie_index]
        movies_list = sorted(list(enumerate(distances)), key=lambda x: x[1], reverse=True)[1:6]

        print(f"🍿 Found closest match: '{matched_title}'")
        print(f"🎬 Top 5 Recommendations for you:\n")
        print("=" * 50)
        for i in movies_list:
            print(f"🎥 {new_df.iloc[i[0]].title}")
        print("=" * 50)

    except Exception as e:
        print(f"❌ Runtime Pipeline Halt: {e}")
        print("💡 Tip: Make sure you ran the cells for CountVectorizer and similarity matrix calculation first!")

# 🚀 Test the updated engine string arrays
recommend("spider man")


❌ 'spider man' not found anywhere in the dataset.

🔍 Let's debug. Here are all titles in your dataset containing 'spider':
['Spider-Man 3' 'The Amazing Spider-Man' 'Spider-Man 2'
 'The Amazing Spider-Man 2' 'Spider-Man' 'The Spiderwick Chronicles'
 'Along Came a Spider' 'Spider' 'Kingdom of the Spiders']


/tmp/ipykernel_530/3592357805.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['title_clean'] = new_df['title'].astype(str).str.strip()


In [16]:
recommend("Avengers")

🍿 Found closest match in database: 'Avengers: Age of Ultron'
🎬 Top 5 Recommendations for your profile:

🎥 Iron Man
🎥 Iron Man 3
🎥 The Avengers
🎥 Captain America: Civil War
🎥 Iron Man 2
